## Data exploration


In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgbm
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
import time
import warnings

warnings.filterwarnings('ignore')

DATA_DIR = "data"

X_train = pd.read_csv(f"{DATA_DIR}/X_train_9xQjqvZ.csv", index_col='ROW_ID')

print(f"X_train shape: {X_train.shape}")
print(f"Unique TS in X_train: {X_train['TS'].nunique()}")
print(f"Unique ALLOCATION in X_train: {X_train['ALLOCATION'].nunique()}")
print("\nGROUP distribution in X_train:")
print(X_train['GROUP'].value_counts(dropna=False))

na_pct = (X_train.isna().mean() * 100)
print("\nTop missing columns in X_train (%):")
print(na_pct[na_pct > 0].sort_values(ascending=False).head(10).round(2))

X_train shape: (527073, 44)
Unique TS in X_train: 2522
Unique ALLOCATION in X_train: 278

GROUP distribution in X_train:
GROUP
3    168913
2    134928
4    123062
1    100170
Name: count, dtype: int64

Top missing columns in X_train (%):
SIGNED_VOLUME_1          73.52
SIGNED_VOLUME_20          1.61
MEDIAN_DAILY_TURNOVER     0.69
SIGNED_VOLUME_19          0.57
SIGNED_VOLUME_18          0.16
RET_20                    0.01
SIGNED_VOLUME_17          0.01
RET_19                    0.01
SIGNED_VOLUME_16          0.01
RET_18                    0.01
dtype: float64


In [ ]:
# Load data
X_train = pd.read_csv(f"{DATA_DIR}/X_train_9xQjqvZ.csv", index_col='ROW_ID')
y_train = pd.read_csv(f"{DATA_DIR}/y_train_Ppwhaz8.csv", index_col='ROW_ID')
X_test  = pd.read_csv(f"{DATA_DIR}/X_test_1zTtEnD.csv", index_col='ROW_ID')

## Phase 2 baseline


In [7]:

#Replicate Exact Benchmark Features
RET_features = [f'RET_{i}' for i in range(1, 21)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]
TURNOVER_features = ['MEDIAN_DAILY_TURNOVER']

for i in [3, 5, 10, 15, 20]:
    X_train[f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(axis=1)
    X_train[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')
    
    X_test[f'AVERAGE_PERF_{i}'] = X_test[RET_features[:i]].mean(axis=1)
    X_test[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_test.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')

for i in [20]:
    X_train[f'STD_PERF_{i}'] = X_train[RET_features[:i]].std(axis=1)
    X_train[f'ALLOCATIONS_STD_PERF_{i}'] = X_train.groupby('TS')[f'STD_PERF_{i}'].transform('mean')
    
    X_test[f'STD_PERF_{i}'] = X_test[RET_features[:i]].std(axis=1)
    X_test[f'ALLOCATIONS_STD_PERF_{i}'] = X_test.groupby('TS')[f'STD_PERF_{i}'].transform('mean')

bench_features = (
    RET_features + SIGNED_VOLUME_features + TURNOVER_features +
    [f'AVERAGE_PERF_{i}' for i in [3, 5, 10, 15, 20]] +
    [f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3, 5, 10, 15, 20]] +
    ['STD_PERF_20', 'ALLOCATIONS_STD_PERF_20']
)

# Add GROUP as categorical for Models 2 & 3
X_train['GROUP_CAT'] = X_train['GROUP'].astype('category')
X_test['GROUP_CAT']  = X_test['GROUP'].astype('category')
phase2_features = bench_features + ['GROUP_CAT']

In [8]:
#Setup Exact 8-Fold Date Split from Benchmark
train_dates = X_train['TS'].unique()
kf = KFold(n_splits=8, random_state=0, shuffle=True)

configs = {
    "1_Bench_MSE": {
        "features": bench_features,
        "target_binary": False,
        "params": {
            "objective": "mse", "metric": "mse", "num_threads": -1,
            "seed": 42, "verbosity": -1, "learning_rate": 1e-2, "max_depth": 3
        }
    },
    "2_Huber_HeavyTail_Group": {
        "features": phase2_features,
        "target_binary": False,
        "params": {
            "objective": "huber", "alpha": 0.003, "metric": "mae", "num_threads": -1,
            "seed": 42, "verbosity": -1, "learning_rate": 1e-2, "max_depth": 3
        }
    },
    "3_Binary_LogLoss_Group": {
        "features": phase2_features,
        "target_binary": True,
        "params": {
            "objective": "binary", "metric": "binary_logloss", "num_threads": -1,
            "seed": 42, "verbosity": -1, "learning_rate": 1e-2, "max_depth": 3
        }
    }
}

In [9]:
#Run Cross-Validation Comparison
results = {}
oof_predictions = {}

for name, cfg in configs.items():
    fold_scores = []
    oof_pred = np.zeros(len(X_train))
    feats = cfg["features"]
    
    for fold, (tr_date_idx, va_date_idx) in enumerate(kf.split(train_dates)):
        tr_mask = X_train['TS'].isin(train_dates[tr_date_idx])
        va_mask = X_train['TS'].isin(train_dates[va_date_idx])
        
        X_tr = X_train.loc[tr_mask, feats]
        X_va = X_train.loc[va_mask, feats]
        
        if cfg["target_binary"]:
            y_tr = (y_train.loc[tr_mask, 'target'] > 0).astype(int).values
            threshold = 0.5
        else:
            y_tr = y_train.loc[tr_mask, 'target'].values
            threshold = 0.0
            
        y_va_true_sign = (y_train.loc[va_mask, 'target'] > 0).astype(int).values
        
        dtrain = lgbm.Dataset(X_tr, label=y_tr)
        model = lgbm.train(cfg["params"], dtrain, num_boost_round=500)
        
        preds = model.predict(X_va)
        oof_pred[va_mask.values] = preds
        
        acc = accuracy_score(y_va_true_sign, (preds > threshold).astype(int))
        fold_scores.append(acc * 100)
        
    mean_acc = np.mean(fold_scores)
    std_acc  = np.std(fold_scores)
    overall_oof_acc = accuracy_score((y_train['target'] > 0).astype(int), (oof_pred > threshold).astype(int)) * 100
    results[name] = (fold_scores, mean_acc, std_acc, overall_oof_acc)
    oof_predictions[name] = oof_pred
    print(f"{name:25s} | Mean Fold Acc: {mean_acc:.2f}% (+/- {std_acc:.2f}%) | Overall OOF Acc: {overall_oof_acc:.2f}%")

# Also check ensembling Huber + Binary OOF
ens_prob = (
    (oof_predictions["2_Huber_HeavyTail_Group"] / np.std(oof_predictions["2_Huber_HeavyTail_Group"])) +
    ((oof_predictions["3_Binary_LogLoss_Group"] - 0.5) / np.std(oof_predictions["3_Binary_LogLoss_Group"]))
)
ens_acc = accuracy_score((y_train['target'] > 0).astype(int), (ens_prob > 0).astype(int)) * 100
print(f"{'4_Ensemble_Huber+Binary':25s} | Overall OOF Acc: {ens_acc:.2f}%")

1_Bench_MSE               | Mean Fold Acc: 52.22% (+/- 0.20%) | Overall OOF Acc: 52.22%
2_Huber_HeavyTail_Group   | Mean Fold Acc: 52.29% (+/- 0.29%) | Overall OOF Acc: 52.29%
3_Binary_LogLoss_Group    | Mean Fold Acc: 52.31% (+/- 0.30%) | Overall OOF Acc: 52.31%
4_Ensemble_Huber+Binary   | Overall OOF Acc: 52.32%


In [10]:
#DIAGNOSTICS & BASELINE SUBMISSION

# Check Phase  Baseline OOF Accuracy by GROUP
y_true_sign = (y_train['target'] > 0).astype(int)
X_train['OOF_BENCH_PRED'] = (oof_predictions["1_Bench_MSE"] > 0).astype(int)
X_train['OOF_P2_ENS_PRED'] = (ens_prob > 0).astype(int)

print("--- PHASE 2 ACCURACY BY GROUP (%) ---")
for g in sorted(X_train['GROUP'].unique()):
    mask = (X_train['GROUP'] == g)
    acc_b = accuracy_score(y_true_sign[mask], X_train.loc[mask, 'OOF_BENCH_PRED']) * 100
    acc_e = accuracy_score(y_true_sign[mask], X_train.loc[mask, 'OOF_P2_ENS_PRED']) * 100
    print(f"GROUP {g} (n={mask.sum():6d}) | Bench MSE: {acc_b:.2f}% | Phase 2 Ensemble: {acc_e:.2f}% (Diff: {acc_e - acc_b:+.2f}%)")

#Generate 8-Fold Averaged Test Predictions for Phase 2 Baseline (Binary + GROUP)
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission_SpGVFuH.csv", index_col='ROW_ID')
test_preds_bin = np.zeros(len(X_test))
test_preds_hub = np.zeros(len(X_test))

for fold, (tr_date_idx, va_date_idx) in enumerate(kf.split(train_dates)):
    tr_mask = X_train['TS'].isin(train_dates[tr_date_idx])
    X_tr = X_train.loc[tr_mask, phase2_features]
    
    # Train Fold Binary Model
    dtrain_bin = lgbm.Dataset(X_tr, label=y_true_sign.loc[tr_mask].values)
    m_bin = lgbm.train(configs["3_Binary_LogLoss_Group"]["params"], dtrain_bin, num_boost_round=500)
    test_preds_bin += m_bin.predict(X_test[phase2_features]) / 8.0
    
    # Train Fold Huber Model
    dtrain_hub = lgbm.Dataset(X_tr, label=y_train.loc[tr_mask, 'target'].values)
    m_hub = lgbm.train(configs["2_Huber_HeavyTail_Group"]["params"], dtrain_hub, num_boost_round=500)
    test_preds_hub += m_hub.predict(X_test[phase2_features]) / 8.0

# Standardized Ensemble on Test Set
test_ens_score = (
    (test_preds_hub / np.std(oof_predictions["2_Huber_HeavyTail_Group"])) +
    ((test_preds_bin - 0.5) / np.std(oof_predictions["3_Binary_LogLoss_Group"]))
)

sub_p2 = pd.DataFrame(
    (test_ens_score > 0).astype(int),
    index=sample_sub.index,
    columns=sample_sub.columns # uses 'prediction' from sample_submission.csv
)
sub_p2.to_csv(f"submissions/sub_phase2_baseline.csv")
print("\nSaved sub_phase2_baseline.csv successfully!")
print("Predicted Positive (1) ratio in X_test:", sub_p2.iloc[:, 0].mean().round(4))

--- PHASE 2 ACCURACY BY GROUP (%) ---
GROUP 1 (n=100170) | Bench MSE: 52.66% | Phase 2 Ensemble: 52.57% (Diff: -0.09%)
GROUP 2 (n=134928) | Bench MSE: 52.37% | Phase 2 Ensemble: 52.46% (Diff: +0.09%)
GROUP 3 (n=168913) | Bench MSE: 51.83% | Phase 2 Ensemble: 52.11% (Diff: +0.28%)
GROUP 4 (n=123062) | Bench MSE: 52.23% | Phase 2 Ensemble: 52.24% (Diff: +0.02%)

Saved sub_phase2_baseline.csv successfully!
Predicted Positive (1) ratio in X_test: 0.618


## Temporal overlap audit


In [11]:
#Build a 40-dimensional Market Regime Signature for each of the 2,522 TS dates in X_train
ret_cols = [f'RET_{i}' for i in range(1, 21)]
ts_mkt_mean = X_train.groupby('TS')[ret_cols].mean().fillna(0)
ts_mkt_std  = X_train.groupby('TS')[ret_cols].std().fillna(0)

# Normalize across dates to test 19-day shifted overlap inside X_train
M_1_19 = ts_mkt_mean[[f'RET_{i}' for i in range(1, 20)]].values
M_2_20 = ts_mkt_mean[[f'RET_{i}' for i in range(2, 21)]].values

M1_norm = (M_1_19 - M_1_19.mean(axis=1, keepdims=True)) / (M_1_19.std(axis=1, keepdims=True) + 1e-12)
M2_norm = (M_2_20 - M_2_20.mean(axis=1, keepdims=True)) / (M_2_20.std(axis=1, keepdims=True) + 1e-12)

# Correlation matrix (2522 x 2522) between Day_i's RET_1..19 and Day_j's RET_2..20
corr_mat = (M1_norm @ M2_norm.T) / 19.0
np.fill_diagonal(corr_mat, -1.0)
max_corrs = corr_mat.max(axis=1)

print("--- X_TRAIN TEMPORAL OVERLAP AUDIT (2,522 DATES) ---")
print(f"Dates with >0.95 shifted 19-day market correlation: {(max_corrs > 0.95).sum()} / {len(max_corrs)}")
print(f"Median best-match correlation across all 2,522 dates: {np.median(max_corrs):.4f}")
print(f"90th percentile best-match correlation:               {np.quantile(max_corrs, 0.90):.4f}")

#Construct 8-Fold Regime/Time-Blocked CV (Clustering 20-day market trajectories so overlapping windows stay in the same fold)
ts_signature = np.hstack([
    (ts_mkt_mean.values - ts_mkt_mean.values.mean(0)) / (ts_mkt_mean.values.std(0) + 1e-8),
    (ts_mkt_std.values  - ts_mkt_std.values.mean(0))  / (ts_mkt_std.values.std(0) + 1e-8)
])
kmeans_blocks = KMeans(n_clusters=8, random_state=42, n_init=20).fit_predict(ts_signature)
ts_to_block = pd.Series(kmeans_blocks, index=ts_mkt_mean.index)
X_train['CV_BLOCK'] = X_train['TS'].map(ts_to_block)

print("\n--- 8-FOLD TIME/REGIME-BLOCKED CV DISTRIBUTION (ROWS PER FOLD) ---")
print(X_train['CV_BLOCK'].value_counts().sort_index())

--- X_TRAIN TEMPORAL OVERLAP AUDIT (2,522 DATES) ---
Dates with >0.95 shifted 19-day market correlation: 2100 / 2522
Median best-match correlation across all 2,522 dates: 0.9935
90th percentile best-match correlation:               0.9988

--- 8-FOLD TIME/REGIME-BLOCKED CV DISTRIBUTION (ROWS PER FOLD) ---
CV_BLOCK
0     37062
1    138897
2      5979
3     70977
4     16502
5    205919
6     38010
7     13727
Name: count, dtype: int64


## Chronological-block CV


In [ ]:


# Build Balanced 8-Fold Chronological-Chain CV from corr_mat
# Greedily chain dates t -> t+1 using highest 19-day shifted correlation
n_dates = len(ts_mkt_mean)
ts_names = ts_mkt_mean.index.to_numpy()

# Find mutual best matches (day i's best next day is j, and day j's best prev day is i)
best_next = corr_mat.argmax(axis=1)
best_prev = corr_mat.argmax(axis=0)
next_map = {}
has_prev = set()

for i in range(n_dates):
    j = best_next[i]
    if best_prev[j] == i and corr_mat[i, j] > 0.80:
        next_map[i] = j
        has_prev.add(j)

# Trace contiguous chronological chains
visited = set()
ordered_indices = []
chain_starts = [i for i in range(n_dates) if i not in has_prev]

for start in chain_starts:
    curr = start
    while curr is not None and curr not in visited:
        visited.add(curr)
        ordered_indices.append(curr)
        curr = next_map.get(curr, None)

# Add any remaining isolated dates
for i in range(n_dates):
    if i not in visited:
        ordered_indices.append(i)

# Slice ordered timeline into 8 equal-sized contiguous time blocks
ordered_ts = ts_names[ordered_indices]
ts_to_time_fold = pd.Series(np.arange(n_dates) * 8 // n_dates, index=ordered_ts)
X_train['TIME_FOLD'] = X_train['TS'].map(ts_to_time_fold)

print("--- BALANCED 8-FOLD CHRONOLOGICAL-BLOCK CV (ROWS PER FOLD) ---")
print(X_train['TIME_FOLD'].value_counts().sort_index())


## Phase 3 features


In [ ]:
# 2. Vectorized Phase 3 Alpha Feature Engineering
def engineer_phase3_features(df):
    df = df.copy()
    ret_cols = [f'RET_{i}' for i in range(1, 21)]
    vol_cols_2_20 = [f'SIGNED_VOLUME_{i}' for i in range(2, 21)]
    
    R = df[ret_cols].values
    R_clean = np.nan_to_num(R, nan=0.0)
    eps = 1e-8
    
    # Block A: Multi-Window Means & Volatility
    for i in [3, 5, 10, 15, 20]:
        df[f'AVERAGE_PERF_{i}'] = df[ret_cols[:i]].mean(axis=1)
        df[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = df.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')
    
    for i in [5, 10, 20]:
        df[f'STD_PERF_{i}'] = df[ret_cols[:i]].std(axis=1)
        df[f'ALLOCATIONS_STD_PERF_{i}'] = df.groupby('TS')[f'STD_PERF_{i}'].transform('mean')
        
    # Block B: Cross-Sectional Z-Scores by TS and Peer-Cluster (TS, GROUP)
    for i in [1, 3, 5, 10, 20]:
        col = 'RET_1' if i == 1 else f'AVERAGE_PERF_{i}'
        ts_mean = df.groupby('TS')[col].transform('mean')
        ts_std  = df.groupby('TS')[col].transform('std')
        df[f'ZSCORE_TS_RET_{i}'] = (df[col] - ts_mean) / (ts_std + eps)
        
        grp_mean = df.groupby(['TS', 'GROUP'])[col].transform('mean')
        grp_std  = df.groupby(['TS', 'GROUP'])[col].transform('std')
        df[f'GROUP_MEAN_RET_{i}'] = grp_mean
        df[f'ZSCORE_GROUP_RET_{i}'] = (df[col] - grp_mean) / (grp_std + eps)
        df[f'GROUP_VS_MKT_RET_{i}'] = grp_mean - ts_mean

    grp_std_mean = df.groupby(['TS', 'GROUP'])['STD_PERF_20'].transform('mean')
    grp_std_std  = df.groupby(['TS', 'GROUP'])['STD_PERF_20'].transform('std')
    df['ZSCORE_GROUP_VOL_20'] = (df['STD_PERF_20'] - grp_std_mean) / (grp_std_std + eps)

    # Block C: Momentum, EMA-MACD, RSI & Lag-1 Return Autocorrelation
    df['MACD_1_5']  = df['RET_1'] - df['AVERAGE_PERF_5']
    df['MACD_3_10'] = df['AVERAGE_PERF_3'] - df['AVERAGE_PERF_10']
    df['MACD_5_20'] = df['AVERAGE_PERF_5'] - df['AVERAGE_PERF_20']
    
    w_fast = np.exp(-np.arange(20) / 3.0); w_fast /= w_fast.sum()
    w_slow = np.exp(-np.arange(20) / 10.0); w_slow /= w_slow.sum()
    df['EMA_FAST_3']  = R_clean @ w_fast
    df['EMA_SLOW_10'] = R_clean @ w_slow
    df['EMA_MACD']    = df['EMA_FAST_3'] - df['EMA_SLOW_10']

    pos_R = np.maximum(R_clean, 0.0)
    abs_R = np.abs(R_clean)
    df['RSI_5']  = pos_R[:, :5].sum(axis=1) / (abs_R[:, :5].sum(axis=1) + eps)
    df['RSI_10'] = pos_R[:, :10].sum(axis=1) / (abs_R[:, :10].sum(axis=1) + eps)
    df['RSI_20'] = pos_R.sum(axis=1) / (abs_R.sum(axis=1) + eps)
    
    df['WIN_RATE_5']  = (R[:, :5] > 0).mean(axis=1)
    df['WIN_RATE_10'] = (R[:, :10] > 0).mean(axis=1)
    df['WIN_RATE_20'] = (R > 0).mean(axis=1)

    r_t   = R_clean[:, :-1]
    r_tm1 = R_clean[:, 1:]
    r_t_c   = r_t - r_t.mean(axis=1, keepdims=True)
    r_tm1_c = r_tm1 - r_tm1.mean(axis=1, keepdims=True)
    df['RET_AUTOCORR_LAG1'] = (r_t_c * r_tm1_c).sum(axis=1) / (
        np.sqrt((r_t_c**2).sum(axis=1) * (r_tm1_c**2).sum(axis=1)) + eps
    )

    # Block D: EVT / Heavy-Tail Risk, Downside Semi-Deviation & CVaR_15%
    neg_R = np.minimum(R_clean, 0.0)
    df['DOWNSIDE_VOL_20']   = np.sqrt((neg_R**2).mean(axis=1))
    df['UPSIDE_VOL_20']     = np.sqrt((pos_R**2).mean(axis=1))
    df['UP_DOWN_VOL_RATIO'] = df['UPSIDE_VOL_20'] / (df['DOWNSIDE_VOL_20'] + eps)
    df['SHARPE_20']         = df['AVERAGE_PERF_20'] / (df['STD_PERF_20'] + eps)
    df['SORTINO_20']        = df['AVERAGE_PERF_20'] / (df['DOWNSIDE_VOL_20'] + eps)
    
    sorted_R = np.sort(R_clean, axis=1)
    df['MIN_RET_20']      = sorted_R[:, 0]
    df['MAX_RET_20']      = sorted_R[:, -1]
    df['CVAR_WORST3_RET'] = sorted_R[:, :3].mean(axis=1)
    df['TAIL_BEST3_RET']  = sorted_R[:, -3:].mean(axis=1)
    df['TAIL_ASYMMETRY']  = df['TAIL_BEST3_RET'] + df['CVAR_WORST3_RET']
    
    z_r = (R_clean - R_clean.mean(axis=1, keepdims=True)) / (R_clean.std(axis=1, keepdims=True) + eps)
    df['SKEW_RET_20'] = (z_r**3).mean(axis=1)
    df['KURT_RET_20'] = (z_r**4).mean(axis=1) - 3.0

    # Block E: Missingness-Proof Liquidity & Turnover Efficiency
    df['SV1_IS_NA']     = df['SIGNED_VOLUME_1'].isna().astype(np.float32)
    df['VOL_MEAN_2_6']  = df[[f'SIGNED_VOLUME_{i}' for i in range(2, 7)]].mean(axis=1)
    df['VOL_MEAN_2_20'] = df[vol_cols_2_20].mean(axis=1)
    df['VOL_STD_2_20']  = df[vol_cols_2_20].std(axis=1)
    df['VOL_TREND']     = df['VOL_MEAN_2_6'] - df['VOL_MEAN_2_20']
    
    grp_to_mean = df.groupby(['TS', 'GROUP'])['MEDIAN_DAILY_TURNOVER'].transform('mean')
    grp_to_std  = df.groupby(['TS', 'GROUP'])['MEDIAN_DAILY_TURNOVER'].transform('std')
    df['ZSCORE_GROUP_TURNOVER'] = (df['MEDIAN_DAILY_TURNOVER'] - grp_to_mean) / (grp_to_std + eps)
    df['RET_PER_TURNOVER_20']   = df['AVERAGE_PERF_20'] / (df['MEDIAN_DAILY_TURNOVER'] + 1e-4)

    df['GROUP_CAT'] = df['GROUP'].astype('category')
    return df

t0 = time.time()
X_train_p3 = engineer_phase3_features(X_train)
X_test_p3  = engineer_phase3_features(X_test)
exclude_cols = {'TS', 'ALLOCATION', 'GROUP', 'CV_BLOCK', 'TIME_FOLD', 'OOF_BENCH_PRED', 'OOF_P2_ENS_PRED'}
phase3_features = [c for c in X_train_p3.columns if c not in exclude_cols]
print(f"\nPhase 3 features engineered: {len(phase3_features)} total features in {time.time() - t0:.1f}s")

In [13]:

y_true_sign = (y_train['target'] > 0).astype(int).values
train_dates = X_train_p3['TS'].unique()

def evaluate_cv(cv_splits, cv_name, feat_list):
    print(f"\n================ {cv_name} ({len(feat_list)} features) ================")
    oof_hub = np.zeros(len(X_train_p3))
    oof_bin = np.zeros(len(X_train_p3))
    models_hub, models_bin = [], []
    
    params_hub = {
        "objective": "huber", "alpha": 0.003, "metric": "mae", "num_threads": -1,
        "seed": 42, "verbosity": -1, "learning_rate": 1e-2, "max_depth": 3, "colsample_bytree": 0.8
    }
    params_bin = {
        "objective": "binary", "metric": "binary_logloss", "num_threads": -1,
        "seed": 42, "verbosity": -1, "learning_rate": 1e-2, "max_depth": 3, "colsample_bytree": 0.8
    }
    
    for fold, (tr_mask, va_mask) in enumerate(cv_splits):
        X_tr, X_va = X_train_p3.loc[tr_mask, feat_list], X_train_p3.loc[va_mask, feat_list]
        
        # 1. Huber Model
        dtr_h = lgbm.Dataset(X_tr, label=y_train.loc[tr_mask, 'target'].values)
        m_h = lgbm.train(params_hub, dtr_h, num_boost_round=500)
        oof_hub[va_mask] = m_h.predict(X_va)
        models_hub.append(m_h)
        
        # 2. Binary Model
        dtr_b = lgbm.Dataset(X_tr, label=y_true_sign[tr_mask])
        m_b = lgbm.train(params_bin, dtr_b, num_boost_round=500)
        oof_bin[va_mask] = m_b.predict(X_va)
        models_bin.append(m_b)
        
    acc_hub = accuracy_score(y_true_sign, (oof_hub > 0.0).astype(int)) * 100
    acc_bin = accuracy_score(y_true_sign, (oof_bin > 0.5).astype(int)) * 100
    
    ens_score = (oof_hub / np.std(oof_hub)) + ((oof_bin - 0.5) / np.std(oof_bin))
    acc_ens = accuracy_score(y_true_sign, (ens_score > 0.0).astype(int)) * 100
    
    print(f"Huber Heavy-Tail OOF Acc:   {acc_hub:.2f}%")
    print(f"Binary Log-Loss OOF Acc:    {acc_bin:.2f}%")
    print(f"Ensemble (Huber+Bin) Acc:   {acc_ens:.2f}%")
    return oof_hub, oof_bin, ens_score, models_hub, models_bin

# Build Split List 1: Benchmark Random-Date 8-Fold CV
kf_rand = KFold(n_splits=8, random_state=0, shuffle=True)
rand_splits = [
    (X_train_p3['TS'].isin(train_dates[tr]).values, X_train_p3['TS'].isin(train_dates[va]).values)
    for tr, va in kf_rand.split(train_dates)
]

# Build Split List 2: Leak-Free Chronological-Block 8-Fold CV
time_splits = [
    ((X_train_p3['TIME_FOLD'] != f).values, (X_train_p3['TIME_FOLD'] == f).values)
    for f in range(8)
]

# Run Evaluations
_, _, _, m_hub_r, m_bin_r = evaluate_cv(rand_splits, "RANDOM-DATE 8-FOLD CV (Phase 3)", phase3_features)
oof_h_t, oof_b_t, ens_t, m_hub_t, m_bin_t = evaluate_cv(time_splits, "CHRONOLOGICAL-BLOCK 8-FOLD CV (Phase 3)", phase3_features)

# Top 15 Features by Gain on Chronological-Block CV
imp = pd.DataFrame(index=phase3_features)
imp['Huber_Gain']  = np.mean([m.feature_importance('gain') for m in m_hub_t], axis=0)
imp['Binary_Gain'] = np.mean([m.feature_importance('gain') for m in m_bin_t], axis=0)
imp['Norm_Gain_%'] = 50 * (imp['Huber_Gain'] / imp['Huber_Gain'].sum() + imp['Binary_Gain'] / imp['Binary_Gain'].sum())
print("\n--- TOP 15 FEATURES BY GAIN (CHRONOLOGICAL-BLOCK CV) ---")
print(imp.sort_values('Norm_Gain_%', ascending=False).head(15).round(2))


================ RANDOM-DATE 8-FOLD CV (Phase 3) (111 features) ================
Huber Heavy-Tail OOF Acc:   52.32%
Binary Log-Loss OOF Acc:    52.41%
Ensemble (Huber+Bin) Acc:   52.43%

================ CHRONOLOGICAL-BLOCK 8-FOLD CV (Phase 3) (111 features) ================
Huber Heavy-Tail OOF Acc:   52.40%
Binary Log-Loss OOF Acc:    52.47%
Ensemble (Huber+Bin) Acc:   52.49%

--- TOP 15 FEATURES BY GAIN (CHRONOLOGICAL-BLOCK CV) ---
                             Huber_Gain  Binary_Gain  Norm_Gain_%
MACD_1_5                           0.29     31730.13        14.99
RET_1                              0.21     16434.42         9.24
ZSCORE_GROUP_TURNOVER              0.09     15507.97         5.98
ALLOCATIONS_STD_PERF_5             0.12     10678.76         5.72
ALLOCATIONS_STD_PERF_20            0.07      7731.08         3.62
ALLOCATIONS_STD_PERF_10            0.05      8172.27         3.28
GROUP_VS_MKT_RET_3                 0.06      6482.37         3.01
ALLOCATIONS_AVERAGE_PERF_3      

## Phase 3.5 interactions


In [14]:
#SECOND-ORDER ALPHA & FEATURE PRUNING 

def add_phase3_5_interactions(df, alloc_stats=None):
    df = df.copy()
    eps = 1e-8
    
    # Ultra-Short Horizon Reversal & Volatility Shock (building on #1 feature MACD_1_5)
    df['MACD_1_3']       = df['RET_1'] - df['AVERAGE_PERF_3']
    df['RET_DIFF_1_2']   = df['RET_1'] - df['RET_2']
    df['RET_DIFF_2_3']   = df['RET_2'] - df['RET_3']
    df['VOL_RATIO_5_20'] = df['STD_PERF_5'] / (df['STD_PERF_20'] + eps)
    df['MKT_VOL_RATIO_5_20'] = df['ALLOCATIONS_STD_PERF_5'] / (df['ALLOCATIONS_STD_PERF_20'] + eps)
    
    #Peer-Group (TS, GROUP) Conditioning of MACD & EVT Tail-Risk Features
    for col in ['MACD_1_5', 'MACD_1_3', 'DOWNSIDE_VOL_20', 'CVAR_WORST3_RET', 'SORTINO_20']:
        g_mean = df.groupby(['TS', 'GROUP'])[col].transform('mean')
        g_std  = df.groupby(['TS', 'GROUP'])[col].transform('std')
        df[f'GROUP_MEAN_{col}']   = g_mean
        df[f'ZSCORE_GROUP_{col}'] = (df[col] - g_mean) / (g_std + eps)
        
    #Turnover-Adjusted Short-Term Reversal Interaction
    df['MACD_1_5_X_TURNOVER'] = df['MACD_1_5'] * df['MEDIAN_DAILY_TURNOVER']
    
    # Target-Free Allocation Intrinsic Style Profile (fit on X_train, mapped to X_test)
    if alloc_stats is None:
        alloc_stats = df.groupby('ALLOCATION').agg(
            ALLOC_HIST_VOL=('STD_PERF_20', 'median'),
            ALLOC_HIST_TO=('MEDIAN_DAILY_TURNOVER', 'median')
        )
    df['ALLOC_HIST_VOL'] = df['ALLOCATION'].map(alloc_stats['ALLOC_HIST_VOL'])
    df['ALLOC_HIST_TO']  = df['ALLOCATION'].map(alloc_stats['ALLOC_HIST_TO'])
    df['VOL_VS_ALLOC_HIST'] = df['STD_PERF_20'] / (df['ALLOC_HIST_VOL'] + eps)
    df['TO_VS_ALLOC_HIST']  = df['MEDIAN_DAILY_TURNOVER'] / (df['ALLOC_HIST_TO'] + 1e-5)
    
    return df, alloc_stats

X_train_p35, alloc_stats = add_phase3_5_interactions(X_train_p3, alloc_stats=None)
X_test_p35, _            = add_phase3_5_interactions(X_test_p3, alloc_stats=alloc_stats)

phase35_features = [c for c in X_train_p35.columns if c not in exclude_cols]
print(f"Total Phase 3.5 Features: {len(phase35_features)}")

# Temporarily bind X_train_p3 to X_train_p35 to reuse evaluate_cv
X_train_p3 = X_train_p35

_, _, ens_r_35, _, _ = evaluate_cv(rand_splits, "RANDOM-DATE 8-FOLD CV (Phase 3.5)", phase35_features)
oof_h_35, oof_b_35, ens_t_35, m_hub_35, m_bin_35 = evaluate_cv(time_splits, "CHRONOLOGICAL-BLOCK 8-FOLD CV (Phase 3.5)", phase35_features)

# Inspect updated Top 20 Features by Gain
imp35 = pd.DataFrame(index=phase35_features)
imp35['Huber_Gain']  = np.mean([m.feature_importance('gain') for m in m_hub_35], axis=0)
imp35['Binary_Gain'] = np.mean([m.feature_importance('gain') for m in m_bin_35], axis=0)
imp35['Norm_Gain_%'] = 50 * (imp35['Huber_Gain'] / imp35['Huber_Gain'].sum() + imp35['Binary_Gain'] / imp35['Binary_Gain'].sum())

print("\n--- TOP 20 PHASE 3.5 FEATURES BY GAIN ---")
print(imp35.sort_values('Norm_Gain_%', ascending=False).head(20).round(2))

# Identify bottom zero/low-gain features for Phase 4 pruning
low_gain_feats = imp35[imp35['Norm_Gain_%'] < 0.05].index.tolist()
print(f"\nFeatures with <0.05% gain (candidates for Phase 4 pruning): {len(low_gain_feats)} / {len(phase35_features)}")

Total Phase 3.5 Features: 131

================ RANDOM-DATE 8-FOLD CV (Phase 3.5) (131 features) ================
Huber Heavy-Tail OOF Acc:   52.45%
Binary Log-Loss OOF Acc:    52.44%
Ensemble (Huber+Bin) Acc:   52.44%

================ CHRONOLOGICAL-BLOCK 8-FOLD CV (Phase 3.5) (131 features) ================
Huber Heavy-Tail OOF Acc:   52.48%
Binary Log-Loss OOF Acc:    52.51%
Ensemble (Huber+Bin) Acc:   52.50%

--- TOP 20 PHASE 3.5 FEATURES BY GAIN ---
                             Huber_Gain  Binary_Gain  Norm_Gain_%
ALLOC_HIST_TO                      0.22     24058.33        10.76
MACD_1_5                           0.18     16796.69         8.29
RET_1                              0.20     13814.09         7.99
ALLOCATIONS_STD_PERF_5             0.10      8142.51         4.25
MACD_1_5_X_TURNOVER                0.05     11609.16         3.76
ALLOCATIONS_STD_PERF_20            0.07      6760.31         3.29
ALLOCATIONS_AVERAGE_PERF_3         0.07      7046.38         3.28
MACD_1_3     

In [15]:
#  GROUP DIAGNOSTICS 

#Compare Phase 3.5 Chronological-Block OOF Accuracy by GROUP vs Phase 2
X_train_p35['OOF_P35_ENS_PRED'] = (ens_t_35 > 0.0).astype(int)
X_train_p35['OOF_P35_BIN_PRED'] = (oof_b_35 > 0.5).astype(int)

print("--- PHASE 3.5 ACCURACY BY GROUP (%) ---")
for g in sorted(X_train_p35['GROUP'].unique()):
    mask = (X_train_p35['GROUP'] == g).values
    acc_p2  = accuracy_score(y_true_sign[mask], X_train.loc[mask, 'OOF_P2_ENS_PRED']) * 100
    acc_p35 = accuracy_score(y_true_sign[mask], X_train_p35.loc[mask, 'OOF_P35_BIN_PRED']) * 100
    print(f"GROUP {g} (n={mask.sum():6d}) | Phase 2: {acc_p2:.2f}% | Phase 3.5: {acc_p35:.2f}% (Gain: {acc_p35 - acc_p2:+.2f}%)")

# Lock in the 91 Active Features (>=0.05% Gain) for Phase 4 Tomorrow
active_phase3_features = imp35[imp35['Norm_Gain_%'] >= 0.05].index.tolist()
print(f"\nLocked in {len(active_phase3_features)} active features (pruned {len(low_gain_feats)} noise features for Phase 4).")

# Generate Phase 3.5 Test Predictions from the 8 Chronological-Block Fold Models
test_preds_hub_p35 = np.mean([m.predict(X_test_p35[phase35_features]) for m in m_hub_35], axis=0)
test_preds_bin_p35 = np.mean([m.predict(X_test_p35[phase35_features]) for m in m_bin_35], axis=0)

test_ens_p35 = (
    (test_preds_hub_p35 / np.std(oof_h_35)) +
    ((test_preds_bin_p35 - 0.5) / np.std(oof_b_35))
)

sub_p3 = pd.DataFrame(
    (test_ens_p35 > 0.0).astype(int),
    index=sample_sub.index,
    columns=sample_sub.columns
)
sub_p3.to_csv(f"submissions/sub_phase3_alpha.csv")
print("Saved sub_phase3_alpha.csv successfully!")
print("Predicted Positive (1) ratio in X_test:", sub_p3.iloc[:, 0].mean().round(4))

--- PHASE 3.5 ACCURACY BY GROUP (%) ---
GROUP 1 (n=100170) | Phase 2: 52.57% | Phase 3.5: 52.33% (Gain: -0.24%)
GROUP 2 (n=134928) | Phase 2: 52.46% | Phase 3.5: 52.80% (Gain: +0.34%)
GROUP 3 (n=168913) | Phase 2: 52.11% | Phase 3.5: 52.34% (Gain: +0.23%)
GROUP 4 (n=123062) | Phase 2: 52.24% | Phase 3.5: 52.57% (Gain: +0.32%)

Locked in 91 active features (pruned 40 noise features for Phase 4).
Saved sub_phase3_alpha.csv successfully!
Predicted Positive (1) ratio in X_test: 0.5967


In [17]:
true_pos_ratio = (y_train['target'] > 0).mean()
print(f"Calibrating to True Positive Ratio: {true_pos_ratio:.4f}")

def calibrate_threshold(oof_preds, target_ratio):
    # Sort predictions and find the threshold that exactly matches the target ratio
    sorted_preds = np.sort(oof_preds)
    split_idx = int(len(sorted_preds) * (1 - target_ratio))
    return sorted_preds[split_idx]

# Calibrate our Phase 3.5 OOF predictions
opt_thresh_hub = calibrate_threshold(oof_h_35, true_pos_ratio)
opt_thresh_bin = calibrate_threshold(oof_b_35, true_pos_ratio)
opt_thresh_ens = calibrate_threshold(ens_t_35, true_pos_ratio)

acc_hub_calib = accuracy_score(y_true_sign, (oof_h_35 > opt_thresh_hub).astype(int)) * 100
acc_bin_calib = accuracy_score(y_true_sign, (oof_b_35 > opt_thresh_bin).astype(int)) * 100
acc_ens_calib = accuracy_score(y_true_sign, (ens_t_35 > opt_thresh_ens).astype(int)) * 100

print(f"\nCALIBRATED PHASE 3.5 ACCURACY")
print(f"Huber Calibrated Acc:    {acc_hub_calib:.2f}% (Threshold: {opt_thresh_hub:.5f})")
print(f"Binary Calibrated Acc:   {acc_bin_calib:.2f}% (Threshold: {opt_thresh_bin:.5f})")
print(f"Ensemble Calibrated Acc: {acc_ens_calib:.2f}% (Threshold: {opt_thresh_ens:.5f})")

Calibrating to True Positive Ratio: 0.5072

CALIBRATED PHASE 3.5 ACCURACY
Huber Calibrated Acc:    52.39% (Threshold: 0.00003)
Binary Calibrated Acc:   52.46% (Threshold: 0.50558)
Ensemble Calibrated Acc: 52.42% (Threshold: 0.36125)


## Phase 4 regularisation


In [18]:
#Hyperparameter Grid Search (Regularization & Handcuffing)
print("\nINITIATING PHASE 4 REGULARIZATION SWEEP")
tuning_grid = [
    {"colsample_bytree": 0.4, "min_data_in_leaf": 1000, "reg_alpha": 0.1, "reg_lambda": 1.0},
    {"colsample_bytree": 0.5, "min_data_in_leaf": 500,  "reg_alpha": 1.0, "reg_lambda": 5.0},
    {"colsample_bytree": 0.6, "min_data_in_leaf": 2000, "reg_alpha": 5.0, "reg_lambda": 10.0}
]

best_acc = 0
best_params = None

for i, params in enumerate(tuning_grid):
    base_params_bin = {
        "objective": "binary", "metric": "binary_logloss", "num_threads": -1,
        "seed": 42, "verbosity": -1, "learning_rate": 0.01, "max_depth": 4
    }
    base_params_bin.update(params)
    
    oof_bin_tune = np.zeros(len(X_train_p35))
    
    for fold, (tr_mask, va_mask) in enumerate(time_splits):
        X_tr, X_va = X_train_p35.loc[tr_mask, active_phase3_features], X_train_p35.loc[va_mask, active_phase3_features]
        y_tr = y_true_sign[tr_mask]
        
        dtr_b = lgbm.Dataset(X_tr, label=y_tr)
        m_b = lgbm.train(base_params_bin, dtr_b, num_boost_round=600)
        oof_bin_tune[va_mask] = m_b.predict(X_va)
        
    tune_thresh = calibrate_threshold(oof_bin_tune, true_pos_ratio)
    tune_acc = accuracy_score(y_true_sign, (oof_bin_tune > tune_thresh).astype(int)) * 100
    
    print(f"Grid {i+1} | {params} | Calibrated Acc: {tune_acc:.2f}%")
    
    if tune_acc > best_acc:
        best_acc = tune_acc
        best_params = params

print(f"\nBest Regularization Parameters Found: {best_params} with {best_acc:.2f}%")


INITIATING PHASE 4 REGULARIZATION SWEEP
Grid 1 | {'colsample_bytree': 0.4, 'min_data_in_leaf': 1000, 'reg_alpha': 0.1, 'reg_lambda': 1.0} | Calibrated Acc: 52.58%
Grid 2 | {'colsample_bytree': 0.5, 'min_data_in_leaf': 500, 'reg_alpha': 1.0, 'reg_lambda': 5.0} | Calibrated Acc: 52.54%
Grid 3 | {'colsample_bytree': 0.6, 'min_data_in_leaf': 2000, 'reg_alpha': 5.0, 'reg_lambda': 10.0} | Calibrated Acc: 52.57%

Best Regularization Parameters Found: {'colsample_bytree': 0.4, 'min_data_in_leaf': 1000, 'reg_alpha': 0.1, 'reg_lambda': 1.0} with 52.58%


In [20]:
#REGULARIZED ENSEMBLE (HUBER + BINARY + RIDGE) & OPTIMAL THRESHOLD
p4_params_bin = {
    "objective": "binary", "metric": "binary_logloss", "num_threads": -1,
    "seed": 42, "verbosity": -1, "learning_rate": 0.01,
    "max_depth": 4, "num_leaves": 12,
    "colsample_bytree": 0.4, "subsample": 0.8, "subsample_freq": 1,
    "min_data_in_leaf": 1500, "reg_alpha": 1.0, "reg_lambda": 5.0
}

p4_params_hub = {
    "objective": "huber", "alpha": 0.003, "metric": "mae", "num_threads": -1,
    "seed": 42, "verbosity": -1, "learning_rate": 0.01,
    "max_depth": 4, "num_leaves": 12,
    "colsample_bytree": 0.4, "subsample": 0.8, "subsample_freq": 1,
    "min_data_in_leaf": 1500, "reg_alpha": 1.0, "reg_lambda": 5.0
}

oof_p4_bin   = np.zeros(len(X_train_p35))
oof_p4_hub   = np.zeros(len(X_train_p35))
oof_p4_ridge = np.zeros(len(X_train_p35))

test_p4_bin   = np.zeros(len(X_test_p35))
test_p4_hub   = np.zeros(len(X_test_p35))
test_p4_ridge = np.zeros(len(X_test_p35))

# Numeric features for Ridge (excluding categorical GROUP_CAT)
ridge_feats = [c for c in active_phase3_features if c != 'GROUP_CAT']
X_tr_ridge_all = X_train_p35[ridge_feats].fillna(0).clip(-1e4, 1e4).values
X_te_ridge_all = X_test_p35[ridge_feats].fillna(0).clip(-1e4, 1e4).values

print("Training Phase 4 Handcuffed Models (Binary + Huber + Ridge) across 8 Chronological Folds...")
for fold, (tr_mask, va_mask) in enumerate(time_splits):
    X_tr = X_train_p35.loc[tr_mask, active_phase3_features]
    X_va = X_train_p35.loc[va_mask, active_phase3_features]
    
    # A. Regularized Binary LightGBM
    dtr_b = lgbm.Dataset(X_tr, label=y_true_sign[tr_mask])
    m_b = lgbm.train(p4_params_bin, dtr_b, num_boost_round=600)
    oof_p4_bin[va_mask] = m_b.predict(X_va)
    test_p4_bin += m_b.predict(X_test_p35[active_phase3_features]) / 8.0
    
    # B. Regularized Huber LightGBM
    dtr_h = lgbm.Dataset(X_tr, label=y_train.loc[tr_mask, 'target'].values)
    m_h = lgbm.train(p4_params_hub, dtr_h, num_boost_round=600)
    oof_p4_hub[va_mask] = m_h.predict(X_va)
    test_p4_hub += m_h.predict(X_test_p35[active_phase3_features]) / 8.0
    
    # C. Heavily Regularized Ridge Regressor
    scaler = RobustScaler()
    X_tr_s = np.clip(scaler.fit_transform(X_tr_ridge_all[tr_mask]), -10, 10)
    X_va_s = np.clip(scaler.transform(X_va_ridge_all := X_tr_ridge_all[va_mask]), -10, 10)
    X_te_s = np.clip(scaler.transform(X_te_ridge_all), -10, 10)
    
    ridge = Ridge(alpha=10000.0)
    ridge.fit(X_tr_s, y_train.loc[tr_mask, 'target'].values)
    oof_p4_ridge[va_mask] = ridge.predict(X_va_s)
    test_p4_ridge += ridge.predict(X_te_s) / 8.0

#Evaluate Individual Models at Natural Bayes Threshold vs Quantile Threshold
acc_bin_nat   = accuracy_score(y_true_sign, (oof_p4_bin > 0.50).astype(int)) * 100
acc_hub_nat   = accuracy_score(y_true_sign, (oof_p4_hub > 0.00).astype(int)) * 100
acc_ridge_nat = accuracy_score(y_true_sign, (oof_p4_ridge > 0.00).astype(int)) * 100

print(f"\n INDIVIDUAL OOF ACCURACY (NATURAL THRESHOLD)")
print(f"P4 Regularized Binary (>0.50): {acc_bin_nat:.2f}%")
print(f"P4 Regularized Huber  (>0.00): {acc_hub_nat:.2f}%")
print(f"P4 Regularized Ridge  (>0.00): {acc_ridge_nat:.2f}%")

z_bin   = (oof_p4_bin - 0.50) / np.std(oof_p4_bin)
z_hub   = oof_p4_hub / np.std(oof_p4_hub)
z_ridge = oof_p4_ridge / np.std(oof_p4_ridge)

oof_p4_blend = 0.45 * z_bin + 0.45 * z_hub + 0.10 * z_ridge
acc_blend_nat = accuracy_score(y_true_sign, (oof_p4_blend > 0.0).astype(int)) * 100

# Sweep threshold in [-0.15, +0.15] to find true OOF-maximizing threshold
ths = np.linspace(-0.15, 0.15, 121)
accs = [accuracy_score(y_true_sign, (oof_p4_blend > t).astype(int)) * 100 for t in ths]
best_t = ths[np.argmax(accs)]
best_blend_acc = np.max(accs)

print(f"\n--- PHASE 4 BLEND OOF ACCURACY ---")
print(f"P4 Blend Natural Threshold (>0.000):  {acc_blend_nat:.2f}% (Pos Ratio: {(oof_p4_blend > 0.0).mean():.4f})")
print(f"P4 Blend Optimal Threshold (>{best_t:+.3f}): {best_blend_acc:.2f}% (Pos Ratio: {(oof_p4_blend > best_t).mean():.4f})")

# 4. Save Final Phase 4 Submission
test_z_bin   = (test_p4_bin - 0.50) / np.std(oof_p4_bin)
test_z_hub   = test_p4_hub / np.std(oof_p4_hub)
test_z_ridge = test_p4_ridge / np.std(oof_p4_ridge)
test_p4_blend = 0.45 * test_z_bin + 0.45 * test_z_hub + 0.10 * test_z_ridge

sub_p4 = pd.DataFrame(
    (test_p4_blend > 0.0).astype(int),
    index=sample_sub.index,
    columns=sample_sub.columns
)
sub_p4.to_csv(f"submissions/sub_phase4_regularized.csv")
print("\nSaved sub_phase4_regularized.csv successfully!")
print("Predicted Positive (1) ratio in X_test:", sub_p4.iloc[:, 0].mean().round(4))

Training Phase 4 Handcuffed Models (Binary + Huber + Ridge) across 8 Chronological Folds...

 INDIVIDUAL OOF ACCURACY (NATURAL THRESHOLD)
P4 Regularized Binary (>0.50): 52.63%
P4 Regularized Huber  (>0.00): 52.61%
P4 Regularized Ridge  (>0.00): 52.12%

--- PHASE 4 BLEND OOF ACCURACY ---
P4 Blend Natural Threshold (>0.000):  52.61% (Pos Ratio: 0.5801)
P4 Blend Optimal Threshold (>-0.092): 52.65% (Pos Ratio: 0.6251)

Saved sub_phase4_regularized.csv successfully!
Predicted Positive (1) ratio in X_test: 0.5794


## Final blend


In [22]:


# 1. Pure Tree Blend (55% P4 Binary + 45% P4 Huber, 0% Ridge)
oof_p4_pure = 0.55 * z_bin + 0.45 * z_hub
acc_pure_nat = accuracy_score(y_true_sign, (oof_p4_pure > 0.0).astype(int)) * 100

ths_p = np.linspace(-0.12, 0.12, 121)
accs_p = [accuracy_score(y_true_sign, (oof_p4_pure > t).astype(int)) * 100 for t in ths_p]
best_t_p = ths_p[np.argmax(accs_p)]
best_pure_acc = np.max(accs_p)

print(f"PURE TREE ENSEMBLE (55% BINARY + 45% HUBER, NO RIDGE)")
print(f"Pure Tree Blend Natural (>0.000):  {acc_pure_nat:.2f}% (Pos Ratio: {(oof_p4_pure > 0.0).mean():.4f})")
print(f"Pure Tree Blend Optimal (>{best_t_p:+.3f}): {best_pure_acc:.2f}% (Pos Ratio: {(oof_p4_pure > best_t_p).mean():.4f})")

# 2. Per-Group Accuracy Audit: Benchmark vs Phase 3.5 vs Phase 4 Pure Trees
X_train_p35['OOF_P4_PURE_PRED'] = (oof_p4_pure > 0.0).astype(int)
print("\nPHASE 4 ACCURACY BY GROUP (%)")
for g in sorted(X_train_p35['GROUP'].unique()):
    mask = (X_train_p35['GROUP'] == g).values
    acc_b   = accuracy_score(y_true_sign[mask], X_train.loc[mask, 'OOF_BENCH_PRED']) * 100
    acc_p35 = accuracy_score(y_true_sign[mask], X_train_p35.loc[mask, 'OOF_P35_BIN_PRED']) * 100
    acc_p4  = accuracy_score(y_true_sign[mask], X_train_p35.loc[mask, 'OOF_P4_PURE_PRED']) * 100
    print(f"GROUP {g} (n={mask.sum():6d}) | Bench: {acc_b:.2f}% | P3.5: {acc_p35:.2f}% | P4 Pure: {acc_p4:.2f}% (vs Bench: {acc_p4 - acc_b:+.2f}%)")

# 3. Export Pure Handcuffed Tree Submission
test_p4_pure = 0.55 * test_z_bin + 0.45 * test_z_hub
sub_p4_pure = pd.DataFrame(
    (test_p4_pure > 0.0).astype(int),
    index=sample_sub.index,
    columns=sample_sub.columns
)
sub_p4_pure.to_csv(f"submissions/sub_phase4_pure_trees.csv")
print("\nSaved sub_phase4_pure_trees.csv successfully!")
print("Predicted Positive (1) ratio in X_test:", sub_p4_pure.iloc[:, 0].mean().round(4))

PURE TREE ENSEMBLE (55% BINARY + 45% HUBER, NO RIDGE)
Pure Tree Blend Natural (>0.000):  52.62% (Pos Ratio: 0.5828)
Pure Tree Blend Optimal (>-0.054): 52.64% (Pos Ratio: 0.6085)

PHASE 4 ACCURACY BY GROUP (%)
GROUP 1 (n=100170) | Bench: 52.66% | P3.5: 52.33% | P4 Pure: 52.74% (vs Bench: +0.08%)
GROUP 2 (n=134928) | Bench: 52.37% | P3.5: 52.80% | P4 Pure: 52.82% (vs Bench: +0.45%)
GROUP 3 (n=168913) | Bench: 51.83% | P3.5: 52.34% | P4 Pure: 52.41% (vs Bench: +0.58%)
GROUP 4 (n=123062) | Bench: 52.23% | P3.5: 52.57% | P4 Pure: 52.58% (vs Bench: +0.35%)

Saved sub_phase4_pure_trees.csv successfully!
Predicted Positive (1) ratio in X_test: 0.5827
